# 🧪 Taller Práctico: Transformers, BERT y RoBERTa

Este notebook acompaña una clase práctica de PLN sobre modelos de lenguaje basados en Transformers. Usaremos modelos preentrenados como BERT y RoBERTa para realizar tareas de análisis de sentimientos, clasificación de texto y análisis de representaciones.

## ✅ Parte 1: Uso de `pipeline` para tareas básicas de NLP

In [2]:
from transformers import pipeline

# Clasificación de sentimiento
task = pipeline("sentiment-analysis")
text = "BERT is a powerfull model for NLP."
result = task(text)
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9991379976272583}]


## ✅ Parte 2: Tokenización y extracción de embeddings

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Transformers are powerful models for NLP."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("Shape del embedding:", outputs.last_hidden_state.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Shape del embedding: torch.Size([1, 10, 768])


## ✅ Parte 3: Clasificación de texto con RoBERTa y dataset AG News

In [6]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score

# Cargar dataset AG News
dataset = load_dataset("ag_news")
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

def preprocess(example):
    return tokenizer(example["text"], truncation=True)

tokenized_ds = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_ds["test"].select(range(200)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_result = trainer.evaluate()
print(eval_result)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

/tmp/ipython-input-2192058882.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hector-gomez (hector-gomez-pontificia-universidad-cat-lica-del-per-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,1.366800
20,1.064900
30,0.634400
40,0.373100
50,0.460900
60,0.308300


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.39035141468048096, 'eval_accuracy': 0.875, 'eval_runtime': 65.2241, 'eval_samples_per_second': 3.066, 'eval_steps_per_second': 0.199, 'epoch': 1.0}


## ✅ Parte 4: Adaptación de RoBERTa en español para clasificación

En esta sección usaremos el modelo `PlanTL-GOB-ES/roberta-base-bne` (RoBERTa entrenado en corpus en español) y un dataset en español para clasificar sentimientos de tweets.

In [7]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification

# Dataset de ejemplo en español
es_dataset = load_dataset("pysentimiento/tweets-sentiment", split="train[:2000]")

model_name = "PlanTL-GOB-ES/roberta-base-bne"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def preprocess_es(example):
    return tokenizer(example['text'], truncation=True)

encoded_dataset = es_dataset.map(preprocess_es, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Métricas
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

DatasetNotFoundError: Dataset 'pysentimiento/tweets-sentiment' doesn't exist on the Hub or cannot be accessed.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()